# 3. Goal Distribution Modeling

Goal:
This notebook trains probabilistic models for predicting home and away goal distributions using ordinal gradient boosted trees.

Note:
This notebook extends the initial feature set by introducing additional football-specific predictors such as rolling statistics, difference features, and Elo-based ratings.

In [ ]:
## 3.1 Import Libraries And Load Data

import numpy as np
import pandas as pd
from pandas.util import hash_pandas_object
import matplotlib.pyplot as plt
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import accuracy_score, log_loss, mean_poisson_deviance, mean_absolute_error


In [ ]:
import pandas as pd

def load_and_merge_seasons(
    seasons,
    data_dir="../data/raw",
    filename_template="EPL {season}.csv",
):
    dfs = []

    for season in seasons:
        path = f"{data_dir}/{filename_template.format(season=season)}"

        d = pd.read_csv(
            path,
            on_bad_lines="skip",   # skip malformed rows
            encoding="utf-8"
        )

        d["Season"] = season
        dfs.append(d)

    df = pd.concat(dfs, ignore_index=True)

    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

    print("Rows with missing dates:", df["Date"].isna().sum())

    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)
    return df

In [3]:
import sys
import os

sys.path.append(os.path.abspath("../src"))
from Feature_Engineering import *
seasons = [
    "93-94", "94-95", "95-96", "96-97", "97-98",
    "98-99", "99-00","00-01",
    "01-02", "02-03", "03-04", "04-05", "05-06",
    "06-07", "07-08", "08-09", "09-10", "10-11",
    "11-12", "12-13", "13-14", "14-15", "15-16",
    "16-17", "17-18", "18-19", "19-20", "20-21",
    "21-22", "22-23", "23-24", "24-25", "25-26"
]
df = load_and_merge_seasons(seasons)
team_df = to_team_level(df)
team_df = add_match_id(team_df)
team_df = add_games_played(team_df)
team_df = add_rolling_features(team_df)

match_df = merge_home_away(team_df)
match_df = clean_match_df(match_df)

C:\Users\user\AppData\Local\Temp\ipykernel_1048\1245242216.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")


Rows with missing dates: 697


In [ ]:
## 3.2 Validate Temporal Structure

print(df[["Date", "Season"]].head(10))
print(df["Date"].isna().sum())
print(df["Date"].min(), df["Date"].max())

        Date Season
0 1993-08-14  93-94
1 1993-08-14  93-94
2 1993-08-14  93-94
3 1993-08-14  93-94
4 1993-08-14  93-94
5 1993-08-14  93-94
6 1993-08-14  93-94
7 1993-08-14  93-94
8 1993-08-14  93-94
9 1993-08-14  93-94
0
1993-08-14 00:00:00 2026-02-02 00:00:00


In [ ]:
## 3.3 Create Additional Engineered Features

match_df["GF_roll3_diff"] = match_df["GF_roll3_home"] - match_df["GF_roll3_away"]
match_df["GA_roll3_diff"] = match_df["GA_roll3_home"] - match_df["GA_roll3_away"]
match_df["Points_roll3_diff"] = match_df["Points_roll3_home"] - match_df["Points_roll3_away"]

match_df["Shots_roll3_diff"] = match_df["Shots_roll3_home"] - match_df["Shots_roll3_away"]
match_df["SoT_roll3_diff"] = match_df["SoT_roll3_home"] - match_df["SoT_roll3_away"]
match_df["Corners_roll3_diff"] = match_df["Corners_roll3_home"] - match_df["Corners_roll3_away"]
match_df["Fouls_roll3_diff"] = match_df["Fouls_roll3_home"] - match_df["Fouls_roll3_away"]
match_df["Yellows_roll3_diff"] = match_df["Yellows_roll3_home"] - match_df["Yellows_roll3_away"]
match_df["Reds_roll3_diff"] = match_df["Reds_roll3_home"] - match_df["Reds_roll3_away"]

match_df["OppShots_roll3_diff"] = match_df["OppShots_roll3_home"] - match_df["OppShots_roll3_away"]
match_df["OppSoT_roll3_diff"] = match_df["OppSoT_roll3_home"] - match_df["OppSoT_roll3_away"]
match_df["OppCorners_roll3_diff"] = match_df["OppCorners_roll3_home"] - match_df["OppCorners_roll3_away"]
match_df["OppFouls_roll3_diff"] = match_df["OppFouls_roll3_home"] - match_df["OppFouls_roll3_away"]
match_df["OppYellows_roll3_diff"] = match_df["OppYellows_roll3_home"] - match_df["OppYellows_roll3_away"]
match_df["OppReds_roll3_diff"] = match_df["OppReds_roll3_home"] - match_df["OppReds_roll3_away"]

match_df["SoT_pct_roll3_diff"] = match_df["SoT_pct_roll3_home"] - match_df["SoT_pct_roll3_away"]
match_df["OppSoT_pct_roll3_diff"] = match_df["OppSoT_pct_roll3_home"] - match_df["OppSoT_pct_roll3_away"]


match_df["GF_roll5_diff"] = match_df["GF_roll5_home"] - match_df["GF_roll5_away"]
match_df["GA_roll5_diff"] = match_df["GA_roll5_home"] - match_df["GA_roll5_away"]
match_df["Points_roll5_diff"] = match_df["Points_roll5_home"] - match_df["Points_roll5_away"]

match_df["Shots_roll5_diff"] = match_df["Shots_roll5_home"] - match_df["Shots_roll5_away"]
match_df["SoT_roll5_diff"] = match_df["SoT_roll5_home"] - match_df["SoT_roll5_away"]
match_df["Corners_roll5_diff"] = match_df["Corners_roll5_home"] - match_df["Corners_roll5_away"]
match_df["Fouls_roll5_diff"] = match_df["Fouls_roll5_home"] - match_df["Fouls_roll5_away"]
match_df["Yellows_roll5_diff"] = match_df["Yellows_roll5_home"] - match_df["Yellows_roll5_away"]
match_df["Reds_roll5_diff"] = match_df["Reds_roll5_home"] - match_df["Reds_roll5_away"]

match_df["OppShots_roll5_diff"] = match_df["OppShots_roll5_home"] - match_df["OppShots_roll5_away"]
match_df["OppSoT_roll5_diff"] = match_df["OppSoT_roll5_home"] - match_df["OppSoT_roll5_away"]
match_df["OppCorners_roll5_diff"] = match_df["OppCorners_roll5_home"] - match_df["OppCorners_roll5_away"]
match_df["OppFouls_roll5_diff"] = match_df["OppFouls_roll5_home"] - match_df["OppFouls_roll5_away"]
match_df["OppYellows_roll5_diff"] = match_df["OppYellows_roll5_home"] - match_df["OppYellows_roll5_away"]
match_df["OppReds_roll5_diff"] = match_df["OppReds_roll5_home"] - match_df["OppReds_roll5_away"]

match_df["SoT_pct_roll5_diff"] = match_df["SoT_pct_roll5_home"] - match_df["SoT_pct_roll5_away"]
match_df["OppSoT_pct_roll5_diff"] = match_df["OppSoT_pct_roll5_home"] - match_df["OppSoT_pct_roll5_away"]


match_df["GF_roll15_diff"] = match_df["GF_roll15_home"] - match_df["GF_roll15_away"]
match_df["GA_roll15_diff"] = match_df["GA_roll15_home"] - match_df["GA_roll15_away"]
match_df["Points_roll15_diff"] = match_df["Points_roll15_home"] - match_df["Points_roll15_away"]

match_df["Shots_roll15_diff"] = match_df["Shots_roll15_home"] - match_df["Shots_roll15_away"]
match_df["SoT_roll15_diff"] = match_df["SoT_roll15_home"] - match_df["SoT_roll15_away"]
match_df["Corners_roll15_diff"] = match_df["Corners_roll15_home"] - match_df["Corners_roll15_away"]
match_df["Fouls_roll15_diff"] = match_df["Fouls_roll15_home"] - match_df["Fouls_roll15_away"]
match_df["Yellows_roll15_diff"] = match_df["Yellows_roll15_home"] - match_df["Yellows_roll15_away"]
match_df["Reds_roll15_diff"] = match_df["Reds_roll15_home"] - match_df["Reds_roll15_away"]

match_df["OppShots_roll15_diff"] = match_df["OppShots_roll15_home"] - match_df["OppShots_roll15_away"]
match_df["OppSoT_roll15_diff"] = match_df["OppSoT_roll15_home"] - match_df["OppSoT_roll15_away"]
match_df["OppCorners_roll15_diff"] = match_df["OppCorners_roll15_home"] - match_df["OppCorners_roll15_away"]
match_df["OppFouls_roll15_diff"] = match_df["OppFouls_roll15_home"] - match_df["OppFouls_roll15_away"]
match_df["OppYellows_roll15_diff"] = match_df["OppYellows_roll15_home"] - match_df["OppYellows_roll15_away"]
match_df["OppReds_roll15_diff"] = match_df["OppReds_roll15_home"] - match_df["OppReds_roll15_away"]

match_df["SoT_pct_roll15_diff"] = match_df["SoT_pct_roll15_home"] - match_df["SoT_pct_roll15_away"]
match_df["OppSoT_pct_roll15_diff"] = match_df["OppSoT_pct_roll15_home"] - match_df["OppSoT_pct_roll15_away"]


match_df["GamesPlayed_diff"] = match_df["GamesPlayed_home"] - match_df["GamesPlayed_away"]

match_df["GD_roll3_home"] = match_df["GF_roll3_home"] - match_df["GA_roll3_home"]
match_df["GD_roll3_away"] = match_df["GF_roll3_away"] - match_df["GA_roll3_away"]
match_df["GD_roll3_diff"] = match_df["GD_roll3_home"] - match_df["GD_roll3_away"]

match_df["GD_roll5_home"] = match_df["GF_roll5_home"] - match_df["GA_roll5_home"]
match_df["GD_roll5_away"] = match_df["GF_roll5_away"] - match_df["GA_roll5_away"]
match_df["GD_roll5_diff"] = match_df["GD_roll5_home"] - match_df["GD_roll5_away"]

match_df["GD_roll15_home"] = match_df["GF_roll15_home"] - match_df["GA_roll15_home"]
match_df["GD_roll15_away"] = match_df["GF_roll15_away"] - match_df["GA_roll15_away"]
match_df["GD_roll15_diff"] = match_df["GD_roll15_home"] - match_df["GD_roll15_away"]

In [ ]:
## 3.4 Elo-Based Team Strength Features

def add_elo_features(
    team_df,
    base_elo=1500.0,
    k=20.0,
    home_advantage=60.0,
):
    """
    Leakage-free Elo on team-level data.

    Assumes:
    - one row per team per match
    - columns: Date, MatchID, Team, Opponent, is_home, FTR
    - FTR uses H / D / A from the original match result

    Adds pre-match Elo features:
    - Elo
    - OppElo
    - EloExp
    - EloDiff

    Important:
    These are PRE-MATCH ratings only.
    Ratings are updated only AFTER both rows of the match receive their pre-match values.
    """
    team_df = team_df.sort_values(["Date", "MatchID", "is_home"], ascending=[True, True, False]).reset_index(drop=True)

    ratings = {}

    elo_vals = np.zeros(len(team_df), dtype=float)
    opp_elo_vals = np.zeros(len(team_df), dtype=float)
    exp_vals = np.zeros(len(team_df), dtype=float)

    # iterate match by match, not row by row
    for match_id, idx in team_df.groupby("MatchID", sort=False).groups.items():
        match_rows = team_df.loc[list(idx)].copy()

        if len(match_rows) != 2:
            raise ValueError(f"MatchID {match_id} does not have exactly 2 rows.")

        home_row = match_rows[match_rows["is_home"] == 1]
        away_row = match_rows[match_rows["is_home"] == 0]

        if len(home_row) != 1 or len(away_row) != 1:
            raise ValueError(f"MatchID {match_id} must have exactly one home row and one away row.")

        home_i = home_row.index[0]
        away_i = away_row.index[0]

        home_team = team_df.at[home_i, "Team"]
        away_team = team_df.at[away_i, "Team"]
        ftr = team_df.at[home_i, "FTR"]  # H / D / A

        home_elo = ratings.get(home_team, base_elo)
        away_elo = ratings.get(away_team, base_elo)

        # expected scores with home advantage applied only in expectation
        home_exp = 1.0 / (1.0 + 10.0 ** ((away_elo - (home_elo + home_advantage)) / 400.0))
        away_exp = 1.0 - home_exp

        # store PRE-MATCH features
        elo_vals[home_i] = home_elo
        opp_elo_vals[home_i] = away_elo
        exp_vals[home_i] = home_exp

        elo_vals[away_i] = away_elo
        opp_elo_vals[away_i] = home_elo
        exp_vals[away_i] = away_exp

        # actual result
        if ftr == "H":
            home_score, away_score = 1.0, 0.0
        elif ftr == "A":
            home_score, away_score = 0.0, 1.0
        elif ftr == "D":
            home_score, away_score = 0.5, 0.5
        else:
            raise ValueError(f"Unexpected FTR value {ftr} for MatchID {match_id}")

        # update AFTER feature assignment
        ratings[home_team] = home_elo + k * (home_score - home_exp)
        ratings[away_team] = away_elo + k * (away_score - away_exp)

    team_df["Elo"] = elo_vals
    team_df["EloExp"] = exp_vals
    # the following features are redundant but i found out that XGBoost benefits from this redundancy so we will keep it
    team_df["OppElo"] = opp_elo_vals
    team_df["EloDiff"] = team_df["Elo"] - team_df["OppElo"]

    return team_df

In [8]:
team_df = add_elo_features(team_df, base_elo=1500, k=20, home_advantage=60)

match_df = merge_home_away(team_df)
match_df = clean_match_df(match_df)


match_df["EloDiff"] = match_df["Elo_home"] - match_df["Elo_away"]
match_df["EloExpDiff"] = match_df["EloExp_home"] - match_df["EloExp_away"]

In [9]:
print(team_df.columns.tolist())
print(match_df.columns.tolist())

['Date', 'Season', 'Team', 'Opponent', 'FTR', 'GF', 'GA', 'HT_GF', 'HT_GA', 'Shots', 'SoT', 'Fouls', 'Corners', 'Yellows', 'Reds', 'OppShots', 'OppSoT', 'OppFouls', 'OppCorners', 'OppYellows', 'OppReds', 'is_home', 'Result', 'Points', 'MatchID', 'GamesPlayed', 'GF_roll3', 'GA_roll3', 'Points_roll3', 'Shots_roll3', 'SoT_roll3', 'Corners_roll3', 'Fouls_roll3', 'Yellows_roll3', 'Reds_roll3', 'OppShots_roll3', 'OppSoT_roll3', 'OppCorners_roll3', 'OppFouls_roll3', 'OppYellows_roll3', 'OppReds_roll3', 'SoT_pct_roll3', 'OppSoT_pct_roll3', 'GF_roll5', 'GA_roll5', 'Points_roll5', 'Shots_roll5', 'SoT_roll5', 'Corners_roll5', 'Fouls_roll5', 'Yellows_roll5', 'Reds_roll5', 'OppShots_roll5', 'OppSoT_roll5', 'OppCorners_roll5', 'OppFouls_roll5', 'OppYellows_roll5', 'OppReds_roll5', 'SoT_pct_roll5', 'OppSoT_pct_roll5', 'GF_roll15', 'GA_roll15', 'Points_roll15', 'Shots_roll15', 'SoT_roll15', 'Corners_roll15', 'Fouls_roll15', 'Yellows_roll15', 'Reds_roll15', 'OppShots_roll15', 'OppSoT_roll15', 'OppCorne

I was thinking of doing gradient descent to find the optimal k and home_advantage parameters for the Elo calculation, using the match outcomes as the target. This would involve defining a loss function based on the difference between predicted probabilities (from EloExp) and actual outcomes, and then optimizing k and home_advantage to minimize this loss.

But it doesn't really matter the initial is like an average and k is somehow related to like the deviation or something like this which could be like scale by the model anyways so it doesn't have to be optimal and we will include in the model home/away feature so that would fix the homeadvantage in the elo if it wasn't optimal 

I chose these values because they were close to elo chess and some other sports and because they were nice round easy numbers

In [ ]:
## 3.5 Expanded Team and Match Features

def add_additional_features(team_df):
    """
    Adds team-level features:
    - Days since last match
    - Goal difference rolling 5
    - Season progress
    """
    team_df = team_df.sort_values(["Team", "Date"]).reset_index(drop=True)

    # Days since last match
    team_df["DaysSinceLastMatch"] = (
        team_df.groupby("Team")["Date"]
        .diff()
        .dt.days
    )
    team_df["DaysSinceLastMatch"] = team_df["DaysSinceLastMatch"].fillna(7)

    # Goal difference rolling 5
    team_df["GD_roll5"] = team_df["GF_roll5"] - team_df["GA_roll5"]

    # Season progress
    team_df["SeasonProgress"] = team_df["GamesPlayed"] / 38

    return team_df

In [ ]:
## 3.6 Matchup-Level Feature Construction

team_df = add_rolling_features(team_df)
team_df = add_additional_features(team_df)

match_df = merge_home_away(team_df)
match_df = clean_match_df(match_df)

In [13]:
# ---------------------------------
# Matchup difference features
# ---------------------------------

# Form difference (recent points)
match_df["FormDiff_roll5"] = (
    match_df["Points_roll5_home"] - match_df["Points_roll5_away"]
)
match_df["EloDiff"] = match_df["Elo_home"] - match_df["Elo_away"]
# Goal difference form
match_df["GD_diff_roll5"] = (
    match_df["GD_roll5_home"] - match_df["GD_roll5_away"]
)
match_df["HomeAdv"] = 1
# Shot volume difference
match_df["Shots_diff_roll5"] = (
    match_df["Shots_roll5_home"] - match_df["Shots_roll5_away"]
)

The plan is : tuning, removing redundant and non effective parameters, tuning again

In [ ]:
## 3.7 Prepare the Modeling Dataset

### time splitting

match_df = match_df.copy()

match_df["Result"] = (
    match_df["Result"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({"W": 0, "D": 1, "L": 2, "H": 0, "A": 2})
)

# drop rows where target could not be mapped
match_df = match_df.dropna(subset=["Result"]).copy()

# make target integer
match_df["Result"] = match_df["Result"].astype(int)

# sort by time
match_df = match_df.sort_values("Date").reset_index(drop=True)

# splits
train = match_df[match_df["Date"] < "2021-07-01"].copy()
val   = match_df[(match_df["Date"] >= "2021-07-01") & (match_df["Date"] < "2023-07-01")].copy()
test  = match_df[match_df["Date"] >= "2023-07-01"].copy()

# targets
y_train = train["Result"]
y_val   = val["Result"]
y_test  = test["Result"]

print("Train size:", len(train))
print("Val size:", len(val))
print("Test size:", len(test))

print("\nTarget distribution:")
print("train\n", y_train.value_counts(dropna=False).sort_index())
print("val\n", y_val.value_counts(dropna=False).sort_index())
print("test\n", y_test.value_counts(dropna=False).sort_index())

Train size: 10714
Val size: 760
Test size: 1000

Target distribution:
train
 Result
0    4918
1    2763
2    3033
Name: count, dtype: int64
val
 Result
0    347
1    175
2    238
Name: count, dtype: int64
test
 Result
0    437
1    239
2    324
Name: count, dtype: int64


In [ ]:
### choosing features and target


raw_postmatch = [
    "GF_home","Shots_home","SoT_home","Fouls_home","Corners_home","Yellows_home","Reds_home","Points_home",
    "GF_away","Shots_away","SoT_away","Fouls_away","Corners_away","Yellows_away","Reds_away","Points_away",
    "Result"
]
feature_cols = [c for c in match_df.columns if (
    "roll" in c or
    "pct" in c or
    "GamesPlayed" in c or
    "Elo" in c or
    "SeasonProgress" in c or
    "DaysSinceLastMatch" in c or
    "GamesPlayed" in c 
)]


X_train, X_val, X_test = train[feature_cols], val[feature_cols], test[feature_cols]
y_train, y_val, y_test = train["Result"], val["Result"], test["Result"]


In [ ]:
## 3.8 Initial Hyperparameter Tuning

from itertools import product
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, log_loss

In [ ]:


# remove inf values if any
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val   = X_val.replace([np.inf, -np.inf], np.nan)

y_train = y_train.astype(int)
y_val   = y_val.astype(int)

# parameter grid
n_estimators_list = 6000
max_depth_list = [2,4,6]
learning_rate_list = [0.01]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [3,5]
gamma_list = [0,0.15,0.3]

results = []

best_logloss = float("inf")
best_params = None
best_model = None

total = len(max_depth_list) * len(learning_rate_list) * len(min_child_weight_list) * len(gamma_list) * len(subsample_list) * len(colsample_list)
run = 0

for max_depth, lr, mcw,g,s,c in product(
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=6000,
        max_depth=max_depth,
        learning_rate=lr,
        subsample=s,
        colsample_bytree=c,
        min_child_weight=mcw,
        random_state=42,
        eval_metric="mlogloss",
        gamma=g,
        early_stopping_rounds=50
    )

    clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

    val_pred = clf.predict(X_val)
    val_proba = clf.predict_proba(X_val)

    acc = accuracy_score(y_val, val_pred)
    ll = log_loss(y_val, val_proba)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll
    })

    print(
        f"[{run}/{total}] "
        f"depth={max_depth} lr={lr} trees={6000} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}"
    )

    if ll < best_logloss:
        best_logloss = ll
        best_params = {
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_model = clf


results_df = pd.DataFrame(results).sort_values("val_logloss")

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))
print(results_df.head(10))


In [ ]:
# parameter grid
n_estimators_list = 6000
max_depth_list = [1,2,3]
learning_rate_list = [0.01]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [4,5,6]
gamma_list = [0.2,0.3]

results = []

best_logloss = float("inf")
best_params = None
best_model = None

total = len(max_depth_list) * len(learning_rate_list) * len(min_child_weight_list) * len(gamma_list) * len(subsample_list) * len(colsample_list)
run = 0

for max_depth, lr, mcw,g,s,c in product(
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=6000,
        max_depth=max_depth,
        learning_rate=lr,
        subsample=s,
        colsample_bytree=c,
        min_child_weight=mcw,
        random_state=42,
        eval_metric="mlogloss",
        gamma=g,
        early_stopping_rounds=50
    )

    clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

    val_pred = clf.predict(X_val)
    val_proba = clf.predict_proba(X_val)

    acc = accuracy_score(y_val, val_pred)
    ll = log_loss(y_val, val_proba)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll
    })

    print(
        f"[{run}/{total}] "
        f"depth={max_depth} lr={lr} trees={6000} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}"
    )

    if ll < best_logloss:
        best_logloss = ll
        best_params = {
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_model = clf


results_df = pd.DataFrame(results).sort_values("val_logloss")

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))
print(results_df.head(10))


In [ ]:
# parameter grid
n_estimators_list = 6000
max_depth_list = [2]
learning_rate_list = [0.001,0.002,0.003,0.004,0.005,0.006,0.007,0.008,0.009,0.01,0.015]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [5]
gamma_list = [0.2]

results = []

best_logloss = float("inf")
best_params = None
best_model = None

total = len(max_depth_list) * len(learning_rate_list) * len(min_child_weight_list) * len(gamma_list) * len(subsample_list) * len(colsample_list)
run = 0

for max_depth, lr, mcw,g,s,c in product(
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=6000,
        max_depth=max_depth,
        learning_rate=lr,
        subsample=s,
        colsample_bytree=c,
        min_child_weight=mcw,
        random_state=42,
        eval_metric="mlogloss",
        gamma=g,
        early_stopping_rounds=50
    )

    clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

    val_pred = clf.predict(X_val)
    val_proba = clf.predict_proba(X_val)

    acc = accuracy_score(y_val, val_pred)
    ll = log_loss(y_val, val_proba)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll
    })

    print(
        f"[{run}/{total}] "
        f"depth={max_depth} lr={lr} trees={6000} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}"
    )

    if ll < best_logloss:
        best_logloss = ll
        best_params = {
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_model = clf


results_df = pd.DataFrame(results).sort_values("val_logloss")

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))
print(results_df.head(10))


In [ ]:
for i in range(1,11):
    print(i/1000,end=",")

In [ ]:
# parameter grid
n_estimators_list = 6000
max_depth_list = [2]
learning_rate_list = [0.006,0.0065,0.007,0.0075,0.0125,0.015,0.0175,0.02,0.03,0.04,0.05]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [5]
gamma_list = [0.2]

results = []

best_logloss = float("inf")
best_params = None
best_model = None

total = len(max_depth_list) * len(learning_rate_list) * len(min_child_weight_list) * len(gamma_list) * len(subsample_list) * len(colsample_list)
run = 0

for max_depth, lr, mcw,g,s,c in product(
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=6000,
        max_depth=max_depth,
        learning_rate=lr,
        subsample=s,
        colsample_bytree=c,
        min_child_weight=mcw,
        random_state=42,
        eval_metric="mlogloss",
        gamma=g,
        early_stopping_rounds=50
    )

    clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

    val_pred = clf.predict(X_val)
    val_proba = clf.predict_proba(X_val)

    acc = accuracy_score(y_val, val_pred)
    ll = log_loss(y_val, val_proba)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll
    })

    print(
        f"[{run}/{total}] "
        f"depth={max_depth} lr={lr} trees={6000} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}"
    )

    if ll < best_logloss:
        best_logloss = ll
        best_params = {
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_model = clf


results_df = pd.DataFrame(results).sort_values("val_logloss")

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))
print(results_df.head(10))


In [ ]:
# parameter grid
n_estimators_list = 6000
max_depth_list = [2]
learning_rate_list = [0.006]
subsample_list = [0.6,0.7,0.8,0.9]
colsample_list = [0.6,0.7,0.8,0.9]
min_child_weight_list = [5]
gamma_list = [0.2]

results = []

best_logloss = float("inf")
best_params = None
best_model = None

total = len(max_depth_list) * len(learning_rate_list) * len(min_child_weight_list) * len(gamma_list) * len(subsample_list) * len(colsample_list)
run = 0

for max_depth, lr, mcw,g,s,c in product(
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=6000,
        max_depth=max_depth,
        learning_rate=lr,
        subsample=s,
        colsample_bytree=c,
        min_child_weight=mcw,
        random_state=42,
        eval_metric="mlogloss",
        gamma=g,
        early_stopping_rounds=50
    )

    clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

    val_pred = clf.predict(X_val)
    val_proba = clf.predict_proba(X_val)

    acc = accuracy_score(y_val, val_pred)
    ll = log_loss(y_val, val_proba)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll
    })

    print(
        f"[{run}/{total}] "
        f"depth={max_depth} lr={lr} trees={6000} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}"
    )

    if ll < best_logloss:
        best_logloss = ll
        best_params = {
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_model = clf


results_df = pd.DataFrame(results).sort_values("val_logloss")

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))
print(results_df.head(10))


In [ ]:
## 3.9 Train the Outcome Model

clf = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    max_depth=2,
    learning_rate=0.006,
    subsample=0.6,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
    n_estimators=6000,
    min_child_weight=5,
    gamma=0.2,
    early_stopping_rounds=50
)

clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

val_pred = clf.predict(X_val)
val_proba = clf.predict_proba(X_val)
print("VAL accuracy:", accuracy_score(y_val, val_pred))
print("VAL logloss:", log_loss(y_val, val_proba))


VAL accuracy: 0.5605263157894737
VAL logloss: 0.9661442525826857


In [19]:
n_estimators = clf.best_iteration + 1
print(n_estimators)

877


In [ ]:
## 3.10 Feature Importance Analysis

imp = pd.Series(clf.feature_importances_, index=feature_cols)
print(imp.sort_values(ascending=False).head(20))

EloDiff                0.059304
EloExp_away            0.046946
EloDiff_home           0.044624
EloExp_home            0.043282
EloDiff_away           0.041618
Shots_diff_roll5       0.015131
Elo_away               0.014764
OppElo_home            0.013808
OppElo_away            0.013378
Elo_home               0.011913
GD_diff_roll5          0.011408
Shots_roll15_away      0.009983
OppSoT_roll15_away     0.009042
OppShots_roll3_away    0.008635
SoT_roll5_home         0.008591
OppShots_roll5_home    0.008492
Points_roll15_away     0.008463
OppShots_roll3_home    0.008427
SeasonProgress_away    0.008427
Points_roll15_home     0.008381
dtype: float32


In [21]:
importances = pd.Series(clf.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=False)

print(importances.head(30))
print(importances.tail(30))

EloDiff                   0.059304
EloExp_away               0.046946
EloDiff_home              0.044624
EloExp_home               0.043282
EloDiff_away              0.041618
Shots_diff_roll5          0.015131
Elo_away                  0.014764
OppElo_home               0.013808
OppElo_away               0.013378
Elo_home                  0.011913
GD_diff_roll5             0.011408
Shots_roll15_away         0.009983
OppSoT_roll15_away        0.009042
OppShots_roll3_away       0.008635
SoT_roll5_home            0.008591
OppShots_roll5_home       0.008492
Points_roll15_away        0.008463
OppShots_roll3_home       0.008427
SeasonProgress_away       0.008427
Points_roll15_home        0.008381
Corners_roll15_away       0.008286
OppCorners_roll5_away     0.007876
OppSoT_roll3_away         0.007838
GF_roll15_away            0.007718
OppCorners_roll15_away    0.007702
Corners_roll5_away        0.007629
GA_roll15_home            0.007615
GD_roll5_away             0.007537
SoT_roll15_home     

In [22]:
importances = pd.Series(clf.feature_importances_, index=X_train.columns).sort_values()
print(importances.head(30))

Reds_roll5_away           0.000000
Yellows_roll3_away        0.004117
Corners_roll3_away        0.004261
OppYellows_roll3_away     0.004343
Yellows_roll5_away        0.004407
OppReds_roll15_home       0.004478
Reds_roll3_home           0.004621
GA_roll3_away             0.004673
OppReds_roll3_away        0.004842
Points_roll5_home         0.004958
SoT_pct_roll5_away        0.004969
OppReds_roll15_away       0.005000
SoT_roll15_away           0.005074
SoT_roll5_away            0.005105
OppSoT_pct_roll3_home     0.005118
OppReds_roll3_home        0.005120
OppYellows_roll15_home    0.005123
OppFouls_roll3_home       0.005178
Yellows_roll3_home        0.005186
Fouls_roll3_home          0.005202
OppSoT_pct_roll5_away     0.005206
Fouls_roll15_away         0.005268
Fouls_roll3_away          0.005268
SoT_pct_roll15_home       0.005284
OppSoT_roll3_home         0.005322
Yellows_roll15_away       0.005335
Fouls_roll15_home         0.005358
Points_roll5_away         0.005396
GF_roll5_home       

In [ ]:
## 3.11 Feature Selection and Model Simplification

drop_features = [
    "Reds_roll5_away"
]

X_train = X_train.drop(columns=drop_features, errors="ignore")
X_val = X_val.drop(columns=drop_features, errors="ignore")
X_test = X_test.drop(columns=drop_features, errors="ignore")

In [24]:
#  training the model
clf = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    max_depth=2,
    learning_rate=0.006,
    subsample=0.6,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
    n_estimators=6000,
    min_child_weight=5,
    gamma=0.2,
    early_stopping_rounds=50
)

clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

val_pred = clf.predict(X_val)
val_proba = clf.predict_proba(X_val)
print("VAL accuracy:", accuracy_score(y_val, val_pred))
print("VAL logloss:", log_loss(y_val, val_proba))


VAL accuracy: 0.5578947368421052
VAL logloss: 0.9658913472425563


In [25]:
importances = pd.Series(clf.feature_importances_, index=X_train.columns).sort_values()
print(importances.head(30))

OppReds_roll5_home         0.000000
OppReds_roll3_home         0.000000
Reds_roll3_home            0.003702
Points_roll5_away          0.004170
OppYellows_roll3_away      0.004527
Yellows_roll3_home         0.004530
Yellows_roll3_away         0.004572
Corners_roll3_away         0.004923
Fouls_roll3_home           0.005079
OppFouls_roll3_home        0.005172
SoT_roll15_away            0.005176
Yellows_roll5_away         0.005221
Reds_roll5_home            0.005225
OppFouls_roll5_home        0.005248
OppSoT_pct_roll3_home      0.005291
GA_roll3_away              0.005314
GF_roll3_away              0.005349
FormDiff_roll5             0.005354
Fouls_roll15_away          0.005396
SoT_pct_roll15_home        0.005406
GF_roll5_home              0.005422
OppReds_roll15_away        0.005427
DaysSinceLastMatch_home    0.005451
SoT_roll5_away             0.005452
OppYellows_roll15_home     0.005459
OppReds_roll15_home        0.005459
Yellows_roll15_away        0.005489
GA_roll5_away              0

In [26]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    clf,
    X_val,
    y_val,
    n_repeats=10,
    random_state=42,
    scoring="neg_log_loss"
)

perm_importance = pd.Series(
    perm_result.importances_mean,
    index=X_val.columns
).sort_values()

print(perm_importance.head(20))

GamesPlayed_away         -0.000629
OppElo_home              -0.000402
OppShots_roll15_away     -0.000373
SoT_pct_roll5_home       -0.000365
OppCorners_roll5_away    -0.000328
GamesPlayed_home         -0.000276
SeasonProgress_away      -0.000206
OppCorners_roll15_away   -0.000196
EloDiff                  -0.000106
Elo_away                 -0.000105
Points_roll3_home        -0.000093
Fouls_roll5_home         -0.000087
GA_roll3_home            -0.000081
Reds_roll15_away         -0.000080
OppSoT_pct_roll5_away    -0.000066
GF_roll15_away           -0.000059
SoT_roll15_away          -0.000059
OppCorners_roll3_home    -0.000052
SeasonProgress_home      -0.000050
Shots_roll3_home         -0.000043
dtype: float64


In [27]:
perm_importance.sort_values(ascending=False).head(20)

EloExp_home              0.031168
EloExp_away              0.001983
OppShots_roll3_away      0.001137
Elo_home                 0.000971
EloDiff_home             0.000920
Shots_diff_roll5         0.000886
OppFouls_roll5_away      0.000353
GD_diff_roll5            0.000312
OppShots_roll5_home      0.000277
GD_roll5_away            0.000250
OppShots_roll3_home      0.000246
OppShots_roll15_home     0.000241
OppCorners_roll3_away    0.000158
OppSoT_roll15_away       0.000154
OppSoT_pct_roll3_away    0.000137
GF_roll15_home           0.000135
OppFouls_roll3_away      0.000133
Points_roll15_home       0.000132
SoT_roll5_home           0.000128
OppYellows_roll3_home    0.000123
dtype: float64

In [28]:
drop_features = [
    "OppShots_roll15_away",
    "OppCorners_roll5_away",
    "OppCorners_roll15_away",
    "OppCorners_roll3_home",
    "Fouls_roll5_home",
    "Reds_roll15_away",
    "Shots_roll3_home"
]
X_train = X_train.drop(columns=drop_features, errors="ignore")
X_val = X_val.drop(columns=drop_features, errors="ignore")
X_test = X_test.drop(columns=drop_features, errors="ignore")

#  training the model
clf = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    max_depth=2,
    learning_rate=0.006,
    subsample=0.6,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
    n_estimators=6000,
    min_child_weight=5,
    gamma=0.2,
    early_stopping_rounds=50
)

clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

val_pred = clf.predict(X_val)
val_proba = clf.predict_proba(X_val)
print("VAL accuracy:", accuracy_score(y_val, val_pred))
print("VAL logloss:", log_loss(y_val, val_proba))
importances = pd.Series(clf.feature_importances_, index=X_train.columns).sort_values()
print(importances.head(30))
perm_result = permutation_importance(
    clf,
    X_val,
    y_val,
    n_repeats=10,
    random_state=42,
    scoring="neg_log_loss"
)

perm_importance = pd.Series(
    perm_result.importances_mean,
    index=X_val.columns
).sort_values()

print(perm_importance.head(20))


VAL accuracy: 0.5605263157894737
VAL logloss: 0.9652131734231452
OppReds_roll5_home         0.000000
OppYellows_roll3_away      0.003883
Reds_roll3_home            0.003943
Points_roll5_away          0.004322
Corners_roll3_away         0.004550
Yellows_roll3_away         0.004938
Yellows_roll3_home         0.005231
Yellows_roll5_away         0.005429
GA_roll5_away              0.005456
SoT_roll15_away            0.005504
GA_roll3_away              0.005561
OppReds_roll15_away        0.005561
Fouls_roll3_home           0.005591
OppYellows_roll15_home     0.005621
OppFouls_roll3_home        0.005631
Fouls_roll15_away          0.005701
Yellows_roll15_away        0.005706
Yellows_roll15_home        0.005716
DaysSinceLastMatch_home    0.005738
Reds_roll3_away            0.005757
SoT_pct_roll15_home        0.005783
OppYellows_roll5_home      0.005824
GF_roll3_away              0.005828
SoT_pct_roll15_away        0.005866
OppSoT_pct_roll5_away      0.005867
OppSoT_pct_roll3_home      0.005874

In [29]:
drop_features = [
    "OppReds_roll5_home"
]
X_train = X_train.drop(columns=drop_features, errors="ignore")
X_val = X_val.drop(columns=drop_features, errors="ignore")
X_test = X_test.drop(columns=drop_features, errors="ignore")

#  training the model
clf = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    max_depth=2,
    learning_rate=0.006,
    subsample=0.6,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
    n_estimators=6000,
    min_child_weight=5,
    gamma=0.2,
    early_stopping_rounds=50
)

clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

val_pred = clf.predict(X_val)
val_proba = clf.predict_proba(X_val)
print("VAL accuracy:", accuracy_score(y_val, val_pred))
print("VAL logloss:", log_loss(y_val, val_proba))
importances = pd.Series(clf.feature_importances_, index=X_train.columns).sort_values()
print(importances.head(30))
perm_result = permutation_importance(
    clf,
    X_val,
    y_val,
    n_repeats=10,
    random_state=42,
    scoring="neg_log_loss"
)

perm_importance = pd.Series(
    perm_result.importances_mean,
    index=X_val.columns
).sort_values()

print(perm_importance.head(20))


VAL accuracy: 0.5592105263157895
VAL logloss: 0.9647499369850359
OppReds_roll3_home        0.000000
OppYellows_roll3_away     0.003039
Corners_roll3_away        0.004533
Points_roll5_away         0.005088
Yellows_roll3_home        0.005277
Yellows_roll3_away        0.005314
GA_roll3_away             0.005420
OppReds_roll15_away       0.005509
OppFouls_roll5_home       0.005521
Yellows_roll5_away        0.005525
OppFouls_roll3_home       0.005566
GF_roll3_away             0.005583
Fouls_roll15_away         0.005620
Reds_roll3_home           0.005630
SoT_roll15_away           0.005638
Yellows_roll15_away       0.005705
SoT_roll3_away            0.005723
OppYellows_roll5_away     0.005726
OppSoT_pct_roll5_home     0.005757
OppSoT_pct_roll3_home     0.005761
Fouls_roll3_home          0.005780
Reds_roll5_home           0.005793
SoT_pct_roll15_away       0.005806
OppSoT_roll3_home         0.005808
Fouls_roll15_home         0.005813
Fouls_roll3_away          0.005830
OppYellows_roll15_home   

In [30]:
drop_features = [
    "OppReds_roll3_home"
]
X_train = X_train.drop(columns=drop_features, errors="ignore")
X_val = X_val.drop(columns=drop_features, errors="ignore")
X_test = X_test.drop(columns=drop_features, errors="ignore")

#  training the model
clf = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    max_depth=2,
    learning_rate=0.006,
    subsample=0.6,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
    n_estimators=6000,
    min_child_weight=5,
    gamma=0.2,
    early_stopping_rounds=50
)

clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

val_pred = clf.predict(X_val)
val_proba = clf.predict_proba(X_val)
print("VAL accuracy:", accuracy_score(y_val, val_pred))
print("VAL logloss:", log_loss(y_val, val_proba))
importances = pd.Series(clf.feature_importances_, index=X_train.columns).sort_values()
print(importances.head(30))
perm_result = permutation_importance(
    clf,
    X_val,
    y_val,
    n_repeats=10,
    random_state=42,
    scoring="neg_log_loss"
)

perm_importance = pd.Series(
    perm_result.importances_mean,
    index=X_val.columns
).sort_values()

print(perm_importance.head(20))


VAL accuracy: 0.5592105263157895
VAL logloss: 0.9644917055768268
OppYellows_roll3_away     0.004497
OppReds_roll3_away        0.004711
GA_roll3_away             0.004846
Corners_roll3_away        0.004910
Points_roll5_away         0.004941
Yellows_roll3_away        0.004959
GF_roll3_away             0.005109
OppSoT_pct_roll3_home     0.005260
Yellows_roll3_home        0.005302
Yellows_roll5_away        0.005369
Yellows_roll15_away       0.005467
OppFouls_roll3_home       0.005489
SoT_roll15_away           0.005523
OppYellows_roll15_home    0.005524
Points_roll5_home         0.005525
OppFouls_roll5_home       0.005543
OppSoT_roll15_home        0.005574
Reds_roll5_home           0.005621
OppReds_roll15_home       0.005645
Reds_roll15_home          0.005651
Fouls_roll3_home          0.005670
OppSoT_pct_roll5_away     0.005672
Fouls_roll15_away         0.005683
SoT_pct_roll5_away        0.005693
Reds_roll3_home           0.005728
OppYellows_roll5_home     0.005740
OppYellows_roll5_away    

In [ ]:
## 3.12 Retuning After Feature Removal

# parameter grid
n_estimators_list = 6000
max_depth_list = [1,2,3,4,5,6]
learning_rate_list = [0.01]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [2,3,4,5,6]
gamma_list = [0,0.1,0.2,0.3]

results = []

best_logloss = float("inf")
best_params = None
best_model = None

total = len(max_depth_list) * len(learning_rate_list) * len(min_child_weight_list) * len(gamma_list) * len(subsample_list) * len(colsample_list)
run = 0

for max_depth, lr, mcw,g,s,c in product(
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=6000,
        max_depth=max_depth,
        learning_rate=lr,
        subsample=s,
        colsample_bytree=c,
        min_child_weight=mcw,
        random_state=42,
        eval_metric="mlogloss",
        gamma=g,
        early_stopping_rounds=50
    )

    clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

    val_pred = clf.predict(X_val)
    val_proba = clf.predict_proba(X_val)

    acc = accuracy_score(y_val, val_pred)
    ll = log_loss(y_val, val_proba)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll
    })

    print(
        f"[{run}/{total}] "
        f"depth={max_depth} lr={lr} trees={6000} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}"
    )

    if ll < best_logloss:
        best_logloss = ll
        best_params = {
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_model = clf


results_df = pd.DataFrame(results).sort_values("val_logloss")

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))
print(results_df.head(10))


[1/120] depth=1 lr=0.01 trees=6000 mcw=2 gamma=0 subsample=0.8 colsample=0.8 -> acc=0.5526, logloss=0.9670
[2/120] depth=1 lr=0.01 trees=6000 mcw=2 gamma=0.1 subsample=0.8 colsample=0.8 -> acc=0.5526, logloss=0.9670
[3/120] depth=1 lr=0.01 trees=6000 mcw=2 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5526, logloss=0.9670
[4/120] depth=1 lr=0.01 trees=6000 mcw=2 gamma=0.3 subsample=0.8 colsample=0.8 -> acc=0.5526, logloss=0.9670
[5/120] depth=1 lr=0.01 trees=6000 mcw=3 gamma=0 subsample=0.8 colsample=0.8 -> acc=0.5526, logloss=0.9670
[6/120] depth=1 lr=0.01 trees=6000 mcw=3 gamma=0.1 subsample=0.8 colsample=0.8 -> acc=0.5526, logloss=0.9670
[7/120] depth=1 lr=0.01 trees=6000 mcw=3 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5526, logloss=0.9670
[8/120] depth=1 lr=0.01 trees=6000 mcw=3 gamma=0.3 subsample=0.8 colsample=0.8 -> acc=0.5526, logloss=0.9670
[9/120] depth=1 lr=0.01 trees=6000 mcw=4 gamma=0 subsample=0.8 colsample=0.8 -> acc=0.5526, logloss=0.9670
[10/120] depth=1 lr=0.01 

In [35]:
#tuning again after removing bad features
# parameter grid
n_estimators_list = 6000
max_depth_list = [2,3]
learning_rate_list = [0.01]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [6,7,8,9,10]
gamma_list = [0.2]

results = []

best_logloss = float("inf")
best_params = None
best_model = None

total = len(max_depth_list) * len(learning_rate_list) * len(min_child_weight_list) * len(gamma_list) * len(subsample_list) * len(colsample_list)
run = 0

for max_depth, lr, mcw,g,s,c in product(
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=6000,
        max_depth=max_depth,
        learning_rate=lr,
        subsample=s,
        colsample_bytree=c,
        min_child_weight=mcw,
        random_state=42,
        eval_metric="mlogloss",
        gamma=g,
        early_stopping_rounds=50
    )

    clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

    val_pred = clf.predict(X_val)
    val_proba = clf.predict_proba(X_val)

    acc = accuracy_score(y_val, val_pred)
    ll = log_loss(y_val, val_proba)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll
    })

    print(
        f"[{run}/{total}] "
        f"depth={max_depth} lr={lr} trees={6000} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}"
    )

    if ll < best_logloss:
        best_logloss = ll
        best_params = {
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_model = clf


results_df = pd.DataFrame(results).sort_values("val_logloss")

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))
print(results_df.head(10))


[1/10] depth=2 lr=0.01 trees=6000 mcw=6 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5566, logloss=0.9665
[2/10] depth=2 lr=0.01 trees=6000 mcw=7 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5579, logloss=0.9664
[3/10] depth=2 lr=0.01 trees=6000 mcw=8 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5566, logloss=0.9663
[4/10] depth=2 lr=0.01 trees=6000 mcw=9 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5566, logloss=0.9661
[5/10] depth=2 lr=0.01 trees=6000 mcw=10 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5566, logloss=0.9661
[6/10] depth=3 lr=0.01 trees=6000 mcw=6 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5605, logloss=0.9674
[7/10] depth=3 lr=0.01 trees=6000 mcw=7 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5605, logloss=0.9671
[8/10] depth=3 lr=0.01 trees=6000 mcw=8 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5605, logloss=0.9669
[9/10] depth=3 lr=0.01 trees=6000 mcw=9 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5605, logloss=0.9663
[10/10] depth=3 lr=0.01 tre

In [36]:
#tuning again after removing bad features
# parameter grid
n_estimators_list = 6000
max_depth_list = [2,3]
learning_rate_list = [0.01]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [10,12,15,20]
gamma_list = [0.2]

results = []

best_logloss = float("inf")
best_params = None
best_model = None

total = len(max_depth_list) * len(learning_rate_list) * len(min_child_weight_list) * len(gamma_list) * len(subsample_list) * len(colsample_list)
run = 0

for max_depth, lr, mcw,g,s,c in product(
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=6000,
        max_depth=max_depth,
        learning_rate=lr,
        subsample=s,
        colsample_bytree=c,
        min_child_weight=mcw,
        random_state=42,
        eval_metric="mlogloss",
        gamma=g,
        early_stopping_rounds=50
    )

    clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

    val_pred = clf.predict(X_val)
    val_proba = clf.predict_proba(X_val)

    acc = accuracy_score(y_val, val_pred)
    ll = log_loss(y_val, val_proba)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll
    })

    print(
        f"[{run}/{total}] "
        f"depth={max_depth} lr={lr} trees={6000} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}"
    )

    if ll < best_logloss:
        best_logloss = ll
        best_params = {
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_model = clf


results_df = pd.DataFrame(results).sort_values("val_logloss")

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))
print(results_df.head(10))


[1/8] depth=2 lr=0.01 trees=6000 mcw=10 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5566, logloss=0.9661
[2/8] depth=2 lr=0.01 trees=6000 mcw=12 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5566, logloss=0.9658
[3/8] depth=2 lr=0.01 trees=6000 mcw=15 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5539, logloss=0.9657
[4/8] depth=2 lr=0.01 trees=6000 mcw=20 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5566, logloss=0.9653
[5/8] depth=3 lr=0.01 trees=6000 mcw=10 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5592, logloss=0.9662
[6/8] depth=3 lr=0.01 trees=6000 mcw=12 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5592, logloss=0.9660
[7/8] depth=3 lr=0.01 trees=6000 mcw=15 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5566, logloss=0.9659
[8/8] depth=3 lr=0.01 trees=6000 mcw=20 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5592, logloss=0.9648

BEST PARAMS:
{'max_depth': 3, 'learning_rate': 0.01, 'min_child_weight': 20, 'gamma': 0.2, 'subsample': 0.8, 'colsample_bytree': 0.8}



In [38]:
#tuning again after removing bad features
# parameter grid
n_estimators_list = 6000
max_depth_list = [3]
learning_rate_list = [0.01]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [18,19,20,21,22,23,24,25,26,27,28,29,30]
gamma_list = [0.2]

results = []

best_logloss = float("inf")
best_params = None
best_model = None

total = len(max_depth_list) * len(learning_rate_list) * len(min_child_weight_list) * len(gamma_list) * len(subsample_list) * len(colsample_list)
run = 0

for max_depth, lr, mcw,g,s,c in product(
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=6000,
        max_depth=max_depth,
        learning_rate=lr,
        subsample=s,
        colsample_bytree=c,
        min_child_weight=mcw,
        random_state=42,
        eval_metric="mlogloss",
        gamma=g,
        early_stopping_rounds=50
    )

    clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

    val_pred = clf.predict(X_val)
    val_proba = clf.predict_proba(X_val)

    acc = accuracy_score(y_val, val_pred)
    ll = log_loss(y_val, val_proba)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll
    })

    print(
        f"[{run}/{total}] "
        f"depth={max_depth} lr={lr} trees={6000} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}"
    )

    if ll < best_logloss:
        best_logloss = ll
        best_params = {
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_model = clf


results_df = pd.DataFrame(results).sort_values("val_logloss")

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))
print(results_df.head(10))


[1/13] depth=3 lr=0.01 trees=6000 mcw=18 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5579, logloss=0.9650
[2/13] depth=3 lr=0.01 trees=6000 mcw=19 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5579, logloss=0.9649
[3/13] depth=3 lr=0.01 trees=6000 mcw=20 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5592, logloss=0.9648
[4/13] depth=3 lr=0.01 trees=6000 mcw=21 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5579, logloss=0.9651
[5/13] depth=3 lr=0.01 trees=6000 mcw=22 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5592, logloss=0.9649
[6/13] depth=3 lr=0.01 trees=6000 mcw=23 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5592, logloss=0.9648
[7/13] depth=3 lr=0.01 trees=6000 mcw=24 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5579, logloss=0.9650
[8/13] depth=3 lr=0.01 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5605, logloss=0.9649
[9/13] depth=3 lr=0.01 trees=6000 mcw=26 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5579, logloss=0.9648
[10/13] depth=3 lr=

In [ ]:
#tuning again after removing bad features
# parameter grid
n_estimators_list = 6000
max_depth_list = [3]
learning_rate_list = [0.001,0.002,0.003,0.004,0.005,0.006,0.007,0.008,0.009,0.01,0.015,0.02,0.03,0.04,0.05]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [25]
gamma_list = [0.2]

results = []

best_logloss = float("inf")
best_params = None
best_model = None

total = len(max_depth_list) * len(learning_rate_list) * len(min_child_weight_list) * len(gamma_list) * len(subsample_list) * len(colsample_list)
run = 0

for max_depth, lr, mcw,g,s,c in product(
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=6000,
        max_depth=max_depth,
        learning_rate=lr,
        subsample=s,
        colsample_bytree=c,
        min_child_weight=mcw,
        random_state=42,
        eval_metric="mlogloss",
        gamma=g,
        early_stopping_rounds=50
    )

    clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

    val_pred = clf.predict(X_val)
    val_proba = clf.predict_proba(X_val)

    acc = accuracy_score(y_val, val_pred)
    ll = log_loss(y_val, val_proba)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll
    })

    print(
        f"[{run}/{total}] "
        f"depth={max_depth} lr={lr} trees={6000} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}"
    )

    if ll < best_logloss:
        best_logloss = ll
        best_params = {
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_model = clf


results_df = pd.DataFrame(results).sort_values("val_logloss")

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))
print(results_df.head(10))


[1/15] depth=3 lr=0.001 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5618, logloss=0.9655
[2/15] depth=3 lr=0.002 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5592, logloss=0.9654
[3/15] depth=3 lr=0.003 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5592, logloss=0.9655
[4/15] depth=3 lr=0.004 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5605, logloss=0.9654
[5/15] depth=3 lr=0.005 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5566, logloss=0.9651
[6/15] depth=3 lr=0.006 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5605, logloss=0.9650
[7/15] depth=3 lr=0.007 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5592, logloss=0.9653
[8/15] depth=3 lr=0.008 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5579, logloss=0.9653
[9/15] depth=3 lr=0.009 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5592, logloss=0.9655
[10/15] de

In [40]:
#tuning again after removing bad features
# parameter grid
n_estimators_list = 6000
max_depth_list = [3]
learning_rate_list = [0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,0.1,0.015,0.025,0.035,0.045,0.055]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [25]
gamma_list = [0.2]

results = []

best_logloss = float("inf")
best_params = None
best_model = None

total = len(max_depth_list) * len(learning_rate_list) * len(min_child_weight_list) * len(gamma_list) * len(subsample_list) * len(colsample_list)
run = 0

for max_depth, lr, mcw,g,s,c in product(
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=6000,
        max_depth=max_depth,
        learning_rate=lr,
        subsample=s,
        colsample_bytree=c,
        min_child_weight=mcw,
        random_state=42,
        eval_metric="mlogloss",
        gamma=g,
        early_stopping_rounds=50
    )

    clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

    val_pred = clf.predict(X_val)
    val_proba = clf.predict_proba(X_val)

    acc = accuracy_score(y_val, val_pred)
    ll = log_loss(y_val, val_proba)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll
    })

    print(
        f"[{run}/{total}] "
        f"depth={max_depth} lr={lr} trees={6000} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}"
    )

    if ll < best_logloss:
        best_logloss = ll
        best_params = {
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_model = clf


results_df = pd.DataFrame(results).sort_values("val_logloss")

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))
print(results_df.head(10))


[1/15] depth=3 lr=0.01 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5605, logloss=0.9649
[2/15] depth=3 lr=0.02 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5605, logloss=0.9654
[3/15] depth=3 lr=0.03 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5566, logloss=0.9650
[4/15] depth=3 lr=0.04 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5500, logloss=0.9646
[5/15] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5605, logloss=0.9646
[6/15] depth=3 lr=0.06 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5566, logloss=0.9634
[7/15] depth=3 lr=0.07 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5579, logloss=0.9639
[8/15] depth=3 lr=0.08 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5526, logloss=0.9643
[9/15] depth=3 lr=0.09 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5500, logloss=0.9671
[10/15] depth=3 lr=

In [41]:
#tuning again after removing bad features
# parameter grid
n_estimators_list = 6000
max_depth_list = [3]
learning_rate_list = [0.05]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [25]
gamma_list = [0,0.1,0.2,0.3]

results = []

best_logloss = float("inf")
best_params = None
best_model = None

total = len(max_depth_list) * len(learning_rate_list) * len(min_child_weight_list) * len(gamma_list) * len(subsample_list) * len(colsample_list)
run = 0

for max_depth, lr, mcw,g,s,c in product(
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=6000,
        max_depth=max_depth,
        learning_rate=lr,
        subsample=s,
        colsample_bytree=c,
        min_child_weight=mcw,
        random_state=42,
        eval_metric="mlogloss",
        gamma=g,
        early_stopping_rounds=50
    )

    clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

    val_pred = clf.predict(X_val)
    val_proba = clf.predict_proba(X_val)

    acc = accuracy_score(y_val, val_pred)
    ll = log_loss(y_val, val_proba)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll
    })

    print(
        f"[{run}/{total}] "
        f"depth={max_depth} lr={lr} trees={6000} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}"
    )

    if ll < best_logloss:
        best_logloss = ll
        best_params = {
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_model = clf


results_df = pd.DataFrame(results).sort_values("val_logloss")

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))
print(results_df.head(10))


[1/4] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.8 colsample=0.8 -> acc=0.5605, logloss=0.9646
[2/4] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0.1 subsample=0.8 colsample=0.8 -> acc=0.5605, logloss=0.9646
[3/4] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0.2 subsample=0.8 colsample=0.8 -> acc=0.5605, logloss=0.9646
[4/4] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0.3 subsample=0.8 colsample=0.8 -> acc=0.5605, logloss=0.9646

BEST PARAMS:
{'max_depth': 3, 'learning_rate': 0.05, 'min_child_weight': 25, 'gamma': 0, 'subsample': 0.8, 'colsample_bytree': 0.8}

TOP RESULTS:
   max_depth  learning_rate  min_child_weight  gamma  subsample  \
0          3           0.05                25    0.0        0.8   
1          3           0.05                25    0.1        0.8   
2          3           0.05                25    0.2        0.8   
3          3           0.05                25    0.3        0.8   

   colsample_bytree  val_accuracy  val_logloss  
0               0.8      0.560526     

In [ ]:
#tuning again after removing bad features
# parameter grid
n_estimators_list = 6000
max_depth_list = [3]
learning_rate_list = [0.05]
subsample_list = [0.6,0.7,0.8,0.9]
colsample_list = [0.6,0.7,0.8,0.9]
min_child_weight_list = [25]
gamma_list = [0]

results = []

best_logloss = float("inf")
best_params = None
best_model = None

total = len(max_depth_list) * len(learning_rate_list) * len(min_child_weight_list) * len(gamma_list) * len(subsample_list) * len(colsample_list)
run = 0

for max_depth, lr, mcw,g,s,c in product(
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=6000,
        max_depth=max_depth,
        learning_rate=lr,
        subsample=s,
        colsample_bytree=c,
        min_child_weight=mcw,
        random_state=42,
        eval_metric="mlogloss",
        gamma=g,
        early_stopping_rounds=50
    )

    clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

    val_pred = clf.predict(X_val)
    val_proba = clf.predict_proba(X_val)

    acc = accuracy_score(y_val, val_pred)
    ll = log_loss(y_val, val_proba)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll
    })

    print(
        f"[{run}/{total}] "
        f"depth={max_depth} lr={lr} trees={6000} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}"
    )

    if ll < best_logloss:
        best_logloss = ll
        best_params = {
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_model = clf


results_df = pd.DataFrame(results).sort_values("val_logloss")

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))
print(results_df.head(10))


[1/16] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.6 colsample=0.6 -> acc=0.5579, logloss=0.9645
[2/16] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.6 colsample=0.7 -> acc=0.5539, logloss=0.9643
[3/16] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.6 colsample=0.8 -> acc=0.5566, logloss=0.9659
[4/16] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.6 colsample=0.9 -> acc=0.5592, logloss=0.9653
[5/16] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.7 colsample=0.6 -> acc=0.5579, logloss=0.9650
[6/16] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.7 colsample=0.7 -> acc=0.5605, logloss=0.9627
[7/16] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.7 colsample=0.8 -> acc=0.5526, logloss=0.9632
[8/16] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.7 colsample=0.9 -> acc=0.5566, logloss=0.9638
[9/16] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.8 colsample=0.6 -> acc=0.5566, logloss=0.9650
[10/16] depth=3 lr=0.05 trees=6000 mc

In [43]:
#tuning again after removing bad features
# parameter grid
n_estimators_list = 6000
max_depth_list = [3]
learning_rate_list = [0.05]
subsample_list = [0.65,0.7,0.75]
colsample_list = [0.65,0.7,0.75]
min_child_weight_list = [25]
gamma_list = [0]

results = []

best_logloss = float("inf")
best_params = None
best_model = None

total = len(max_depth_list) * len(learning_rate_list) * len(min_child_weight_list) * len(gamma_list) * len(subsample_list) * len(colsample_list)
run = 0

for max_depth, lr, mcw,g,s,c in product(
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    clf = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=6000,
        max_depth=max_depth,
        learning_rate=lr,
        subsample=s,
        colsample_bytree=c,
        min_child_weight=mcw,
        random_state=42,
        eval_metric="mlogloss",
        gamma=g,
        early_stopping_rounds=50
    )

    clf.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
    )

    val_pred = clf.predict(X_val)
    val_proba = clf.predict_proba(X_val)

    acc = accuracy_score(y_val, val_pred)
    ll = log_loss(y_val, val_proba)

    results.append({
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll
    })

    print(
        f"[{run}/{total}] "
        f"depth={max_depth} lr={lr} trees={6000} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}"
    )

    if ll < best_logloss:
        best_logloss = ll
        best_params = {
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_model = clf


results_df = pd.DataFrame(results).sort_values("val_logloss")

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))
print(results_df.head(10))


[1/9] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.65 colsample=0.65 -> acc=0.5526, logloss=0.9639
[2/9] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.65 colsample=0.7 -> acc=0.5566, logloss=0.9648
[3/9] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.65 colsample=0.75 -> acc=0.5539, logloss=0.9643
[4/9] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.7 colsample=0.65 -> acc=0.5605, logloss=0.9638
[5/9] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.7 colsample=0.7 -> acc=0.5605, logloss=0.9627
[6/9] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.7 colsample=0.75 -> acc=0.5553, logloss=0.9625
[7/9] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.75 colsample=0.65 -> acc=0.5579, logloss=0.9657
[8/9] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.75 colsample=0.7 -> acc=0.5605, logloss=0.9648
[9/9] depth=3 lr=0.05 trees=6000 mcw=25 gamma=0 subsample=0.75 colsample=0.75 -> acc=0.5579, logloss=0.9635

BEST PARAMS:
{'max_depth': 3, 'le

In [47]:
clf = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=6000,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=25,
    subsample=0.7,
    colsample_bytree=0.7,
    gamma=0,
    random_state=42,
    eval_metric="mlogloss",
    early_stopping_rounds=50
)

clf.fit(
X_train,
y_train,
eval_set=[(X_val, y_val)],
verbose=False
)

print(clf.best_iteration)

84


In [48]:
#final model with best parameters
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

clf_final = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=84,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=25,
    subsample=0.7,
    colsample_bytree=0.7,
    gamma=0,
    random_state=42,
    eval_metric="mlogloss"
)
        
clf_final.fit(
    X_train_final,
    y_train_final
    )


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=0,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=25, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=84, n_jobs=None, num_class=3, ...)

In [49]:
from sklearn.metrics import accuracy_score, log_loss, classification_report
from sklearn.calibration import CalibratedClassifierCV
test_proba = clf_final.predict_proba(X_test)
test_pred = test_proba.argmax(axis=1)
print("TEST accuracy:", accuracy_score(y_test, test_pred))
print("TEST logloss:", log_loss(y_test, test_proba))
print(classification_report(y_test, test_pred))

TEST accuracy: 0.538
TEST logloss: 0.9775206818894367
              precision    recall  f1-score   support

           0       0.54      0.82      0.65       437
           1       0.00      0.00      0.00       239
           2       0.53      0.56      0.54       324

    accuracy                           0.54      1000
   macro avg       0.36      0.46      0.40      1000
weighted avg       0.41      0.54      0.46      1000



c:\Users\user\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\user\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\user\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [50]:
calibrator = CalibratedClassifierCV(clf_final, method="sigmoid", cv="prefit")
calibrator.fit(X_val, y_val)

CalibratedClassifierCV(cv='prefit',
                       estimator=XGBClassifier(base_score=None, booster=None,
                                               callbacks=None,
                                               colsample_bylevel=None,
                                               colsample_bynode=None,
                                               colsample_bytree=0.7,
                                               device=None,
                                               early_stopping_rounds=None,
                                               enable_categorical=False,
                                               eval_metric='mlogloss',
                                               feature_types=None,
                                               feature_weights=None, gamma=0,
                                               grow_policy=None,
                                               importance_type=None,
                                               interaction_constraints=None,
                                               learning_rate=0.05, max_bin=None,
                                               max_cat_threshold=None,
                                               max_cat_to_onehot=None,
                                               max_delta_step=None, max_depth=3,
                                               max_leaves=None,
                                               min_child_weight=25, missing=nan,
                                               monotone_constraints=None,
                                               multi_strategy=None,
                                               n_estimators=84, n_jobs=None,
                                               num_class=3, ...))

In [52]:
import joblib
#saving model
joblib.dump(clf_final, "xgb_final_model.pkl")
joblib.dump(feature_cols, "feature_columns.pkl")

['feature_columns.pkl']

In [53]:
#feature importance
importances = pd.Series(
    clf_final.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importances.head(20))

Elo_away               0.056435
EloExp_away            0.055237
EloExp_home            0.049919
EloDiff                0.045775
EloDiff_away           0.045628
EloDiff_home           0.044317
OppElo_home            0.021722
Elo_home               0.016754
Shots_diff_roll5       0.012439
OppElo_away            0.011478
GD_diff_roll5          0.011320
Shots_roll15_away      0.009026
SoT_roll5_home         0.008741
OppShots_roll3_away    0.008677
Corners_roll5_home     0.008528
OppSoT_roll15_away     0.008230
GA_roll15_home         0.008194
Corners_roll15_away    0.008103
OppSoT_roll3_away      0.008064
GA_roll15_away         0.008033
dtype: float32


In [ ]:
# if we sum all of the elo we get 0.347265

In [ ]:
#exemple prediction
example = X_test.iloc[:10]
pred = clf_final.predict_proba(example)

pd.DataFrame(pred, columns=["AwayWin", "Draw", "HomeWin"])

,AwayWin,Draw,HomeWin
0,0.136016,0.147676,0.716308
1,0.608613,0.227181,0.164206
2,0.476679,0.284055,0.239266
3,0.345317,0.266975,0.387708
4,0.260413,0.207004,0.532583
5,0.757084,0.164521,0.078395
6,0.530248,0.288595,0.181157
7,0.197462,0.237281,0.565257
8,0.421300,0.288685,0.290015
9,0.707539,0.184562,0.107898


In [ ]:
# making it a binary model only


In [55]:
# Remove draws
train_mask = y_train != 1
val_mask = y_val != 1
test_mask = y_test != 1

X_train_bin = X_train[train_mask]
y_train_bin = y_train[train_mask]

X_val_bin = X_val[val_mask]
y_val_bin = y_val[val_mask]

X_test_bin = X_test[test_mask]
y_test_bin = y_test[test_mask]

In [56]:
y_train_bin = (y_train_bin == 0).astype(int)
y_val_bin = (y_val_bin == 0).astype(int)
y_test_bin = (y_test_bin == 0).astype(int)

In [57]:
from xgboost import XGBClassifier

clf_bin = XGBClassifier(
    objective="binary:logistic",
    n_estimators=6000,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=25,
    subsample=0.7,
    colsample_bytree=0.7,
    gamma=0,
    random_state=42,
    eval_metric="logloss",
    early_stopping_rounds=50
)

clf_bin.fit(
    X_train_bin,
    y_train_bin,
    eval_set=[(X_val_bin, y_val_bin)],
    verbose=False
)
print(clf_bin.best_iteration)

95


In [58]:
from sklearn.metrics import accuracy_score, log_loss, classification_report

test_pred_bin = clf_bin.predict(X_test_bin)
test_proba_bin = clf_bin.predict_proba(X_test_bin)

print("Binary TEST accuracy:", accuracy_score(y_test_bin, test_pred_bin))
print("Binary TEST logloss:", log_loss(y_test_bin, test_proba_bin))

print(classification_report(y_test_bin, test_pred_bin))

Binary TEST accuracy: 0.7017082785808147
Binary TEST logloss: 0.5626636169915246
              precision    recall  f1-score   support

           0       0.69      0.54      0.61       324
           1       0.71      0.82      0.76       437

    accuracy                           0.70       761
   macro avg       0.70      0.68      0.68       761
weighted avg       0.70      0.70      0.69       761



In [60]:
#downloading precessed data 

match_df.to_csv("match_df.csv", index=False)

print("File saved as match_df")


team_df.to_csv("team_df.csv", index=False)

print("File saved as team_df")

File saved as match_df
File saved as team_df


In [61]:
match_df.head()

,GF_home,GA_home,HT_GF_home,HT_GA_home,Shots_home,SoT_home,Fouls_home,Corners_home,Yellows_home,Reds_home,...,AwayTeam,Date,Season,FTR,Result,FormDiff_roll5,EloDiff,GD_diff_roll5,HomeAdv,Shots_diff_roll5
0,0.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Coventry,1993-08-14,93-94,A,2,NaN,0.0,NaN,1,NaN
1,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Tottenham,1993-08-14,93-94,A,2,NaN,0.0,NaN,1,NaN
2,0.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Ipswich,1993-08-14,93-94,A,2,NaN,0.0,NaN,1,NaN
3,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Blackburn,1993-08-14,93-94,A,2,NaN,0.0,NaN,1,NaN
4,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Sheffield Weds,1993-08-14,93-94,H,0,NaN,0.0,NaN,1,NaN


## 3.14 Interpretation

This notebook shows that football outcome prediction benefits strongly from structured feature engineering. Rolling team statistics, matchup differences, and Elo-based ratings improved the quality of the feature space beyond the simple baseline.

Feature selection also showed that not all engineered variables contribute equally. Removing weak or redundant variables helped simplify the model without sacrificing much predictive performance.

Overall, this notebook serves as the bridge between the baseline outcome model and the later probabilistic goal-distribution approach.